# Modified GBM calibration & Monte Carlo — period 2014

**Period file:** **2014-01-01 → 2014-12-31**.

| Role | Ticker |
|------|--------|
| Primary | **SPY** |
| Secondary | AAPL |
| Secondary | MSFT |

**Section roles**
- **§4 Calibration only:** choose lookback / rolling, **Reestimate**, inspect estimated parameters (no Monte Carlo plots here).
- **§5 Monte Carlo only:** Start / Restart; simulated paths and history comparison.
- **§6 Optimal stopping:** after §4 (and §5 paths), LSM exercise decision on SPY American calls (risk-neutral paths from the same simulator); results in `stopping_results`.

True rolling rule: at each update, re-estimate the four transition probabilities and \((\mu_U,\sigma_U,\mu_D,\sigma_D)\) from the current window and use them for the next MC segment. See `ROLLING_CALIBRATION.md`.



## 0. Setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA = Path("..") / ".." / "data"
PERIOD_START = pd.Timestamp("2014-01-01")
PERIOD_END = pd.Timestamp("2014-12-31")
TICKERS = ["AAPL", "MSFT", "SPY"]
N_DAYS = 252
N_STEPS = 500  # Monte Carlo time steps per path
COLORS = {"AAPL": "#1f77b4", "MSFT": "#ff7f0e", "SPY": "#2ca02c"}

WINDOW_OPTIONS = {
    "3 months": pd.DateOffset(months=3),
    "6 months": pd.DateOffset(months=6),
    "1 year": pd.DateOffset(years=1),
    "2 years": pd.DateOffset(years=2),
    "3 years": pd.DateOffset(years=3),
    "5 years": pd.DateOffset(years=5),
}
ROLLING_OPTIONS = ["daily", "monthly", "none"]

prices = pd.read_csv(DATA / "equity" / "prices_clean.csv", parse_dates=["Date"]).set_index("Date").sort_index()
period_prices = prices.loc[PERIOD_START:PERIOD_END, TICKERS].copy()
log_returns_all = np.log(prices[TICKERS]).diff()

rolling = {}
cal_meta = {}

print(f"Price sample: {prices.index.min().date()} → {prices.index.max().date()}")
print(
    f"Period rows: {len(period_prices)} trading days "
    f"({period_prices.index.min().date()} → {period_prices.index.max().date()})"
)
period_prices.head()

# --- clean plotting / widget memory (important after reopen) ---
plt.close("all")
plt.ioff()


## 1. Stock price trends (2014)

Adjusted close for AAPL, MSFT, and SPY (primary).


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for ax, ticker in zip(axes, TICKERS):
    s = period_prices[ticker].dropna()
    ax.plot(s.index, s.values, color=COLORS[ticker], lw=1.4)
    role = "primary" if ticker == "SPY" else "secondary"
    ax.set_ylabel("Adj close")
    ax.set_title(f"{ticker} ({role}) — adjusted close, 2014")
axes[-1].set_xlabel("Date")
fig.suptitle("Stock price trends — period 2014", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
display(period_prices.describe().T[["count", "mean", "min", "max"]].round(4))


## 2. Strike prices in this period

Unique strikes \(K\) from `*_options_panel.csv` with `trading_date` in **2014-01-01 → 2014-12-31**.

> AAPL strikes are on the option/contract scale; equity adj closes are split-adjusted.


In [ ]:
for ticker in TICKERS:
    path = DATA / "options" / "processed" / f"{ticker}_options_panel.csv"
    opt = pd.read_csv(path, usecols=["trading_date", "K"], parse_dates=["trading_date"])
    m = (opt["trading_date"] >= PERIOD_START) & (opt["trading_date"] <= PERIOD_END)
    sub = opt.loc[m]
    uniq = np.sort(sub["K"].dropna().unique())
    dmin, dmax = sub["trading_date"].min(), sub["trading_date"].max()
    display(Markdown(
        f"### {ticker} — {len(uniq)} unique strikes "
        f"(options quotes {dmin.date() if pd.notna(dmin) else 'n/a'} → "
        f"{dmax.date() if pd.notna(dmax) else 'n/a'})"
    ))
    print("Strikes K:", ", ".join(f"{x:g}" for x in uniq))
    display(pd.DataFrame({"K": uniq}).T)


## 3. Estimation formulas (Modified GBM)

A three-stage discrete replacement for GBM. Direction is a two-state Markov chain; magnitude is normal and taken positive; the signed return drives the price.

**Stage 1 — direction.** Let \(U=\{r_t>0\}\) and \(D=\{r_t<0\}\). From consecutive non-zero returns in the lookback window, calibrate

\[
\hat P(U\mid U),\ \hat P(D\mid U),\ \hat P(U\mid D),\ \hat P(D\mid D)
\]

(Laplace-smoothed). The previous bar's direction selects which pair is used, so a high \(\hat P(U\mid U)\) or \(\hat P(D\mid D)\) produces upward or downward clustering.

**Stage 2 — magnitude.** After the direction is chosen, draw the size from a separate normal for up and down moves and take it positive:

\[
m_t\mid U \sim \bigl|N(\mu_U,\sigma_U^2)\bigr|,\qquad
m_t\mid D \sim \bigl|N(\mu_D,\sigma_D^2)\bigr|.
\]

\(\mu_U,\sigma_U\) (resp. \(\mu_D,\sigma_D\)) are the sample mean and standard deviation of \(|r|\) on up (resp. down) bars, on the same scale as the series (daily or 1-minute).

**Stage 3 — price.** The signed return is \(r_t=+m_t\) on an up move and \(r_t=-m_t\) on a down move:

\[
S_{t+1}=S_t\,e^{r_t}.
\]

| Parameter | Role |
|-----------|------|
| \(P(U\mid U),\ P(D\mid U)\) | Next-bar direction after an up move |
| \(P(U\mid D),\ P(D\mid D)\) | Next-bar direction after a down move |
| \(\mu_U,\sigma_U\) | Mean / SD of up magnitudes |
| \(\mu_D,\sigma_D\) | Mean / SD of down magnitudes |

Risk-neutral paths (§6) keep the calibrated transitions and magnitudes, then shift each step so \(E[e^{r_t}]=\exp(r\,\Delta t)\).

**True rolling:** at each update date, re-estimate from the lookback window ending there; those params drive the next Monte Carlo segment.



## 4. Calibration only — 2014

Sliders + **Reestimate**. Shows **only** the rolling parameter graphs (no tables).  
Monte Carlo vs history is in **§5** only — one pair per company.



In [ ]:
def estimate_modified_gbm(log_rets: pd.Series):
    """Markov direction + split-normal magnitudes on the bar's log returns."""
    x = log_rets.dropna().astype(float)
    x = x[np.isfinite(x)]
    n = int(x.shape[0])
    nz = x[x != 0.0]
    if n < 3 or int(nz.shape[0]) < 3:
        return None
    signed = nz.to_numpy(dtype=float)
    up = signed > 0.0
    mag = np.abs(signed)
    prev, curr = up[:-1], up[1:]
    n_from_u = int(prev.sum())
    n_from_d = int((~prev).sum())
    n_uu = int((prev & curr).sum())
    n_dd = int((~prev & ~curr).sum())
    p_uu = (n_uu + 0.5) / (n_from_u + 1.0)
    p_dd = (n_dd + 0.5) / (n_from_d + 1.0)
    p_du = 1.0 - p_uu
    p_ud = 1.0 - p_dd

    def _mu_sig(arr):
        """Folded-normal match: E[|N(μ,σ)|] = sample mean of |R|."""
        import math
        if arr.size >= 2:
            m = float(arr.mean())
            s = float(arr.std(ddof=1))
        elif arr.size == 1:
            m = float(arr[0])
            s = float(np.median(mag)) if mag.size else 1e-6
        else:
            m = float(np.median(mag)) if mag.size else 1e-6
            s = m
        if not np.isfinite(m) or m < 0:
            m = abs(m) if np.isfinite(m) else 1e-6
        if not np.isfinite(s) or s <= 0:
            s = 1e-6

        def _eabs(mu, sig):
            sig = max(float(sig), 1e-12)
            a = float(mu) / sig
            return sig * math.sqrt(2.0 / math.pi) * math.exp(-0.5 * a * a) + float(mu) * math.erf(
                a / math.sqrt(2.0)
            )

        e0 = _eabs(0.0, s)
        if e0 >= m:
            mu = 0.0
            sig = m / max(math.sqrt(2.0 / math.pi), 1e-12)
        else:
            lo, hi = 0.0, max(8.0 * m, 1e-6)
            for _ in range(48):
                mid = 0.5 * (lo + hi)
                if _eabs(mid, s) < m:
                    lo = mid
                else:
                    hi = mid
            mu, sig = 0.5 * (lo + hi), s
        if not np.isfinite(sig) or sig <= 0:
            sig = 1e-6
        if not np.isfinite(mu) or mu < 0:
            mu = 0.0
        return float(mu), float(sig)

    mu_u, sig_u = _mu_sig(mag[up])
    mu_d, sig_d = _mu_sig(mag[~up])
    return {
        "n_days": n,
        "p_uu": float(p_uu),
        "p_du": float(p_du),
        "p_ud": float(p_ud),
        "p_dd": float(p_dd),
        "mu_u": mu_u,
        "sig_u": sig_u,
        "mu_d": mu_d,
        "sig_d": sig_d,
        "last_up": 1.0 if bool(up[-1]) else 0.0,
        "p_u": float(up.mean()),
    }


def _slice_window(rets: pd.Series, end: pd.Timestamp, offset: pd.DateOffset) -> pd.Series:
    start = end - offset
    return rets.loc[(rets.index > start) & (rets.index <= end)]


def calibrate_ticker(ticker: str, window_label: str, rolling_mode: str) -> pd.DataFrame:
    rets = log_returns_all[ticker].dropna()
    offset = WINDOW_OPTIONS[window_label]
    rows = []

    if rolling_mode == "daily":
        update_dates = rets.loc[(rets.index >= PERIOD_START) & (rets.index <= PERIOD_END)].index
    elif rolling_mode == "monthly":
        t0 = period_prices[ticker].dropna().index[0]
        month_ends = pd.date_range(PERIOD_START, PERIOD_END, freq="ME")
        update_dates = pd.DatetimeIndex([t0]).append(month_ends).unique().sort_values()
    else:
        update_dates = pd.DatetimeIndex([period_prices[ticker].dropna().index[0]])

    for t_u in update_dates:
        window = _slice_window(rets, pd.Timestamp(t_u), offset)
        est = estimate_modified_gbm(window)
        if est is None:
            continue
        rows.append({
            "date": pd.Timestamp(t_u),
            "window_start": window.index.min(),
            "window_end": window.index.max(),
            **est,
        })
    return pd.DataFrame(rows)


def _mc_time_grid(hist: pd.Series, n_steps: int = N_STEPS):
    """Evenly spaced trading-day grid with exactly n_steps steps (n_steps+1 prices)."""
    hist = hist.dropna()
    n_full = len(hist)
    if n_full <= n_steps + 1:
        return hist
    idx = np.linspace(0, n_full - 1, n_steps + 1)
    idx = np.rint(idx).astype(int)
    for i in range(1, len(idx)):
        if idx[i] <= idx[i - 1]:
            idx[i] = min(idx[i - 1] + 1, n_full - 1)
    return hist.iloc[idx]

def param_schedule_for_steps(ticker: str, cal_table: pd.DataFrame):
    hist = _mc_time_grid(period_prices[ticker], N_STEPS)
    dates = hist.index
    n_steps = len(dates) - 1
    cal = cal_table.sort_values("date").reset_index(drop=True)
    cal_dates = pd.to_datetime(cal["date"]).to_numpy()
    cols = ["p_uu", "p_du", "p_ud", "p_dd", "mu_u", "sig_u", "mu_d", "sig_d", "last_up"]
    arrs = {c: cal[c].to_numpy(dtype=float) for c in cols}
    steps = {c: np.empty(n_steps, dtype=float) for c in cols}
    for i in range(n_steps):
        idx = np.searchsorted(cal_dates, np.datetime64(dates[i]), side="right") - 1
        if idx < 0:
            idx = 0
        for c in cols:
            steps[c][i] = arrs[c][idx]
    return dates, steps, float(hist.iloc[0]), hist

def _show_fig(fig):
    """Show a figure exactly once as PNG.

    Root cause of duplicates: with %matplotlib inline, display(fig) inside an
    Output can ALSO be flushed again by the inline backend → same graph twice
    (worse after reopen when the old kernel is still alive). Saving PNG bytes
    and closing the Figure first avoids that second paint.
    """
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def plot_rolling_paths(rolling_dict: dict, window_label: str, rolling_mode: str):
    """Rolling Modified GBM meanfix parameters (graphs only)."""
    panels = [
        (("p_uu", "p_dd"), r"$\hat P(U|U)$ / $\hat P(D|D)$", "Direction persistence"),
        (("p_ud", "p_du"), r"$\hat P(U|D)$ / $\hat P(D|U)$", "Direction reversal"),
        (("mu_u", "mu_d"), r"$\hat\mu_U$ / $\hat\mu_D$", "Magnitude means (per bar)"),
        (("sig_u", "sig_d"), r"$\hat\sigma_U$ / $\hat\sigma_D$", "Magnitude SDs (per bar)"),
    ]
    with plt.ioff():
        fig, axes = plt.subplots(4, 1, figsize=(11, 11), sharex=True)
        for ax, ((c1, c2), ylab, title) in zip(axes, panels):
            for t in TICKERS:
                r = rolling_dict[t]
                x = pd.to_datetime(r["date"])
                mark = "o" if len(r) < 40 else None
                ax.plot(x, r[c1], lw=1.2, label=f"{t} {c1}", color=COLORS[t], marker=mark, ms=3)
                ax.plot(x, r[c2], lw=1.0, ls="--", color=COLORS[t], alpha=0.75, marker=mark, ms=3)
            ax.set_ylabel(ylab)
            ax.set_title(f"{title} — {rolling_mode}, lookback {window_label}")
            ax.legend(frameon=False, ncol=3, fontsize=8)
        axes[-1].set_xlabel("Date")
        fig.tight_layout()
    _show_fig(fig)

def reestimate(_=None):
    global rolling, cal_meta
    window_label = window_slider.value
    rolling_mode = rolling_slider.value
    rolling = {t: calibrate_ticker(t, window_label, rolling_mode) for t in TICKERS}
    cal_meta = {"window_label": window_label, "rolling_mode": rolling_mode}

    with cal_out:
        clear_output(wait=True)
        display(Markdown(
            f"**Calibration updated:** lookback=`{window_label}`, rolling=`{rolling_mode}` "
            f"(n_updates: " + ", ".join(f"{t}={len(rolling[t])}" for t in TICKERS) + ")"
        ))
        plot_rolling_paths(rolling, window_label, rolling_mode)
        display(Markdown("Go to **§5** and click **Start** for one MC pair per company."))


btn_reestimate.on_click(reestimate)
display(cal_ui)
reestimate()



## 5. Monte Carlo only — one graph pair per company (2014)

| Left | Right |
|------|--------|
| Monte Carlo paths + median | Median path + 25–75% band vs historical prices |

Uses latest **Reestimate** from §4. **Start** / **Restart** redraw that single pair (never stacks another copy).

**Stock-path metrics** (printed under each pair)

1. **MAE** — median absolute error of the 50th percentile path vs actual $S_t$ (central-tendency fit).
2. **ICP** — interval coverage probability: share of actual prices that fall inside the 25th–75th percentile band.
3. **Average band width** — mean($p_{75}-p_{25}$); how narrow or wide the model’s uncertainty range is.



In [ ]:
def simulate_modified_gbm_rolling(steps, S0, n_paths, seed, rf=None):
    """Three-stage Modified GBM: Markov direction, split-normal size, S * exp(r).

    `rf` is an annual risk-free rate. When set, each step is shifted so
    E[exp(r_t)] = exp(rf / N_DAYS). Leave `rf=None` for P-measure paths.
    """
    rng = np.random.default_rng(seed)
    n_steps = len(steps["p_uu"])
    paths = np.empty((n_paths, n_steps + 1), dtype=float)
    paths[:, 0] = S0
    up = np.full(n_paths, float(steps["last_up"][0]) >= 0.5, dtype=bool)
    rf_step = None if rf is None else float(rf) / float(N_DAYS)

    for i in range(n_steps):
        p_uu = float(np.clip(steps["p_uu"][i], 0.0, 1.0))
        p_ud = float(np.clip(steps["p_ud"][i], 0.0, 1.0))
        mu_u = float(steps["mu_u"][i])
        mu_d = float(steps["mu_d"][i])
        sig_u = max(float(steps["sig_u"][i]), 1e-12)
        sig_d = max(float(steps["sig_d"][i]), 1e-12)
        p_up = np.where(up, p_uu, p_ud)
        up = rng.random(n_paths) < p_up
        mag = np.empty(n_paths, dtype=float)
        n_up = int(np.count_nonzero(up))
        n_dn = n_paths - n_up
        if n_up:
            mag[up] = np.abs(rng.normal(mu_u, sig_u, size=n_up))
        if n_dn:
            mag[~up] = np.abs(rng.normal(mu_d, sig_d, size=n_dn))
        mag = np.maximum(mag, 1e-16)
        r = np.where(up, mag, -mag)
        if rf_step is not None:
            mx = float(np.mean(np.exp(r)))
            r = r + (rf_step - np.log(max(mx, 1e-300)))
        paths[:, i + 1] = paths[:, i] * np.exp(r)
    return paths

def _show_fig(fig):
    """Show a figure exactly once as PNG.

    Root cause of duplicates: with %matplotlib inline, display(fig) inside an
    Output can ALSO be flushed again by the inline backend → same graph twice
    (worse after reopen when the old kernel is still alive). Saving PNG bytes
    and closing the Figure first avoids that second paint.
    """
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def _draw_ticker_pair(ticker: str, out: widgets.Output, seed: int, n_paths: int = 1000):
    """Replace contents of `out` with exactly one 1×2 figure."""
    with out:
        clear_output(wait=True)
        if ticker not in rolling or len(rolling[ticker]) == 0:
            display(Markdown("Run **Reestimate** in §4 first."))
            return
        dates_now, steps_now, S0_now, hist_now = param_schedule_for_steps(
            ticker, rolling[ticker]
        )
        paths = simulate_modified_gbm_rolling(steps_now, S0_now, n_paths, seed)
        expected = paths.mean(axis=0)
        p25 = np.percentile(paths, 25, axis=0)
        p50 = np.percentile(paths, 50, axis=0)
        p75 = np.percentile(paths, 75, axis=0)
        _hist = np.asarray(hist_now.values, dtype=float)
        _n = min(len(p50), len(_hist))
        p25, p50, p75, expected, _hist = p25[:_n], p50[:_n], p75[:_n], expected[:_n], _hist[:_n]

        with plt.ioff():
            fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
            x = dates_now[:_n]
            axes[0].plot(x, paths[:, :_n].T, color=COLORS[ticker], alpha=0.12, lw=0.7)
            axes[0].plot(x, p50, color="black", lw=2.0, ls="--", label="median path (50th)")
            axes[0].set_title(f"{ticker}: Monte Carlo")
            axes[0].set_ylabel("price")
            axes[0].legend(loc="best", frameon=False)

            axes[1].fill_between(x, p25, p75, color=COLORS[ticker], alpha=0.18, lw=0, zorder=1, label="25–75% range")
            axes[1].plot(x, _hist, color=COLORS[ticker], lw=1.8, label="historical", zorder=3)
            axes[1].plot(x, p50, color="black", lw=2.0, ls="--", label="median path (50th)", zorder=4)
            axes[1].set_title(f"{ticker}: median vs history")
            axes[1].set_ylabel("price")
            axes[1].legend(loc="best", frameon=False)
            for ax in axes:
                ax.set_xlabel("date")
            mae = float(np.mean(np.abs(p50 - _hist)))
            icp = float(np.mean((_hist >= p25) & (_hist <= p75)))
            abw = float(np.mean(p75 - p25))
            rmse = float(100.0 * np.sqrt(np.mean(((p50 - _hist) / np.maximum(np.abs(_hist), 1e-8)) ** 2)))
            fig.suptitle(
                f"{ticker} | MAE={mae:.4f} | ICP={100*icp:.1f}% | width={abw:.4f} | seed={seed} | "
                f"{cal_meta.get('rolling_mode')} / {cal_meta.get('window_label')}",
                fontsize=11,
                y=1.02,
            )
            fig.tight_layout()
        _show_fig(fig)
        display(Markdown(
            f"**MAE (50th vs $S_t$)** = `{mae:.4f}` · "
            f"**ICP (25–75)** = `{100*icp:.1f}%` · "
            f"**avg band width** = `{abw:.4f}` · "
            f"RMSE%(p50) = `{rmse:.2f}%` | seed = `{seed}`"
        ))


def make_ticker_panel(ticker: str, n_paths: int = 1000):
    """One Output per company. Start/Restart only replace that Output (no stacking)."""
    state = {"seed": 42}
    mode = cal_meta.get("rolling_mode", "?")
    win = cal_meta.get("window_label", "?")
    out = widgets.Output(layout=widgets.Layout(width="100%"))
    btn_start = widgets.Button(description="Start", button_style="success", icon="play")
    btn_restart = widgets.Button(description="Restart", button_style="warning", icon="refresh")
    info = widgets.HTML(f"<b>{ticker}</b> — one graph pair | lookback={win}, mode={mode}")

    busy = {"on": False}

    def on_start(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    def on_restart(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            state["seed"] = int(np.random.default_rng().integers(0, 1_000_000_000))
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    btn_start.on_click(on_start)
    btn_restart.on_click(on_restart)
    return widgets.VBox([info, widgets.HBox([btn_start, btn_restart]), out])


plt.close("all")
plt.ioff()

mc_host = widgets.VBox([])
children = [widgets.HTML("<b>§5 Monte Carlo — click <i>Start</i> once per company (one pair only)</b>")]
for ticker in TICKERS:
    role = "primary" if ticker == "SPY" else "secondary"
    children.append(widgets.HTML(f"<h4 style='margin:8px 0 4px'>{ticker} ({role})</h4>"))
    children.append(make_ticker_panel(ticker))
mc_host.children = tuple(children)
display(mc_host)



## 6. Optimal stopping (American calls — Modified GBM)

Continuous with §5: after Monte Carlo stock paths are available, use the **same Modified GBM simulator** and §4 calibration on **SPY / AAPL / MSFT** to decide exercise vs wait for American calls.

**Do not average simulated stock paths before stopping decisions.** Generate a cloud of individual risk-neutral paths (each keeps its own shocks). Then Longstaff–Schwartz:

1. At each exercise date, compute the immediate payoff $\max(S_t-K,0)$ **on every path**.
2. Estimate continuation by regression on in-the-money simulated states ($1, S, S^2$, path vol proxy).
3. **Each path** exercises iff payoff $>$ continuation; otherwise it continues.
4. Discount that path's stopping payoff to $t=0$.
5. The American value is the **average of those discounted payoffs** (then $\max$ with the $t=0$ intrinsic).

Paths for pricing are **risk-neutral** (drift $\mu \rightarrow r$ from the option panel; vol/jumps from §4 for that ticker). §5's expected-vs-history plot is visualization only — it is not the input to LSM.

**Workflow:** §4 **Reestimate** → §5 **Start** (optional viz) → §6 **Compute stopping** (all three underlyings).




In [ ]:
import sys
_SCRIPTS = Path("..") / "scripts"
if str(_SCRIPTS.resolve()) not in sys.path:
    sys.path.insert(0, str(_SCRIPTS.resolve()))

from american_lsm import (
    STOP_TICKERS,
    lsm_american_call,
    load_calls,
    params_asof,
    sample_calls,
)

def _show_fig(fig):
    """Show figure once as PNG (same pattern as §5)."""
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def _rn_paths_for_contract(row, n_paths: int, seed: int):
    """Risk-neutral paths to expiry using §5 simulator (mean-corrected E[e^{r_t}]=e^{r Δt})."""
    ticker = str(getattr(row, "underlying", "SPY")).upper()
    if ticker not in rolling or len(rolling[ticker]) == 0:
        raise RuntimeError(f"No {ticker} calibration — run Reestimate in §4 first.")
    p = params_asof(rolling[ticker], row.trading_date)
    if p is None:
        raise RuntimeError(f"No {ticker} calibration — run Reestimate in §4 first.")
    dte = int(row.dte)
    if dte < 2:
        raise ValueError("dte must be >= 2")
    n_steps = int(getattr(row, "n_steps", 0)) or int(dte)
    r = float(row.r)
    S0 = float(row.S_t)
    steps = {
        "p_uu": np.full(n_steps, float(p["p_uu"]), dtype=float),
        "p_du": np.full(n_steps, float(p["p_du"]), dtype=float),
        "p_ud": np.full(n_steps, float(p["p_ud"]), dtype=float),
        "p_dd": np.full(n_steps, float(p["p_dd"]), dtype=float),
        "mu_u": np.full(n_steps, float(p["mu_u"]), dtype=float),
        "sig_u": np.full(n_steps, float(p["sig_u"]), dtype=float),
        "mu_d": np.full(n_steps, float(p["mu_d"]), dtype=float),
        "sig_d": np.full(n_steps, float(p["sig_d"]), dtype=float),
        "last_up": np.full(n_steps, float(p["last_up"]), dtype=float),
    }
    return simulate_modified_gbm_rolling(steps, S0, n_paths, seed, rf=r)


_STOP_TICKERS = list(STOP_TICKERS)
_contracts_by_ticker = {}
for _t in _STOP_TICKERS:
    _panel = load_calls(DATA, _t)
    _contracts_by_ticker[_t] = sample_calls(
        _panel, PERIOD_START, PERIOD_END, 
    )

stopping_results = {}  # ticker -> DataFrame

for _t in _STOP_TICKERS:
    _n = len(_contracts_by_ticker[_t])
    display(Markdown(
        f"Sampled **{_n}** {_t} American calls in "
        f"{PERIOD_START.date()} → {PERIOD_END.date()} "
        f"(one nearest-ATM call each Monday, or the next session if Monday is closed; DTE 7–60)."
    ))
    if _n:
        display(
            _contracts_by_ticker[_t][
                ["trading_date", "S_t", "K", "dte", "r", "moneyness", "option_price"]
            ].head(8)
        )

_stop_n_paths = widgets.IntSlider(
    value=2000, min=500, max=8000, step=500, description="n_paths",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="360px"),
)
_stop_seed = widgets.IntText(value=42, description="seed", layout=widgets.Layout(width="200px"))
_btn_stop = widgets.Button(
    description="Compute stopping", button_style="primary", icon="calculator"
)
_stop_out = widgets.Output(layout=widgets.Layout(width="100%"))
_stop_busy = {"on": False}


def _run_optimal_stopping(_=None):
    global stopping_results
    if _stop_busy["on"]:
        return
    _stop_busy["on"] = True
    with _stop_out:
        clear_output(wait=True)
        try:
            missing = [t for t in _STOP_TICKERS if t not in rolling or len(rolling[t]) == 0]
            if missing:
                display(Markdown(
                    "Run **Reestimate** in §4 first "
                    f"(need calibration for: {', '.join(missing)})."
                ))
                return

            n_paths = int(_stop_n_paths.value)
            seed0 = int(_stop_seed.value)
            dt = 1.0 / 252.0  # trading-day clock for LSM; never the 1-min N_DAYS
            stopping_results = {}

            for ticker in _STOP_TICKERS:
                contracts = _contracts_by_ticker[ticker]
                if contracts is None or len(contracts) == 0:
                    display(Markdown(f"No {ticker} call contracts in this period panel slice."))
                    continue

                rows = []
                example = None
                for i, row in enumerate(contracts.itertuples(index=False)):
                    paths = _rn_paths_for_contract(row, n_paths, seed0 + i)
                    res = lsm_american_call(paths, K=float(row.K), r=float(row.r), dt=dt)
                    err = res.price - float(row.option_price)
                    rows.append({
                        "ticker": ticker,
                        "trading_date": row.trading_date,
                        "S_t": float(row.S_t),
                        "K": float(row.K),
                        "dte": int(row.dte),
                        "r": float(row.r),
                        "market": float(row.option_price),
                        "model_price": res.price,
                        "error": err,
                        "early_ex_frac": res.early_exercise_frac,
                        "mean_ex_day": res.mean_exercise_step,
                    })
                    if example is None:
                        example = (row, paths, res)

                df = pd.DataFrame(rows)
                stopping_results[ticker] = df
                rmse = float(100.0 * np.sqrt(np.mean((df["error"] / np.maximum(np.abs(df["market"]), 1e-8)) ** 2)))
                mae = float(np.mean(np.abs(df["error"])))
                color = COLORS.get(ticker, "#2ca02c")

                display(Markdown(
                    f"### Modified GBM — LSM results ({ticker})\n"
                    f"n_paths={n_paths} | contracts={len(df)} | "
                    f"RMSE={rmse:.2f}% | MAE={mae:.4f} | "
                    f"mean early-exercise fraction="
                    f"{df['early_ex_frac'].mean():.3f}"
                ))
                display(
                    df[
                        ["trading_date", "S_t", "K", "dte", "market", "model_price",
                         "error", "early_ex_frac", "mean_ex_day"]
                    ].round(4)
                )

                with plt.ioff():
                    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
                    ax = axes[0]
                    ax.scatter(df["market"], df["model_price"], alpha=0.75, color=color)
                    lo = min(df["market"].min(), df["model_price"].min())
                    hi = max(df["market"].max(), df["model_price"].max())
                    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
                    ax.set_xlabel("market option_price")
                    ax.set_ylabel("model LSM price")
                    ax.set_title("Price: model vs market")

                    axes[1].bar(
                        ["model", "market"],
                        [df["model_price"].mean(), df["market"].mean()],
                        color=[color, "#7f7f7f"],
                    )
                    axes[1].set_title("Mean option value")
                    axes[1].set_ylabel("price")

                    axes[2].hist(
                        df["mean_ex_day"], bins=12,
                        color=color, alpha=0.85, edgecolor="white",
                    )
                    axes[2].set_xlabel("mean exercise day (by contract)")
                    axes[2].set_title("Optimal exercise timing")
                    fig.suptitle(
                        f"Modified GBM optimal stopping | {ticker} | "
                        f"{cal_meta.get('rolling_mode')} / {cal_meta.get('window_label')}",
                        fontsize=11, y=1.02,
                    )
                    fig.tight_layout()
                _show_fig(fig)

                if example is not None:
                    row, paths, res = example
                    j = int(np.argmin(np.abs(res.exercise_steps - res.mean_exercise_step)))
                    t_ex = int(res.exercise_steps[j])
                    with plt.ioff():
                        fig2, ax = plt.subplots(figsize=(10, 3.8))
                        ax.plot(paths[j], color=color, lw=1.5, label="one RN path")
                        ax.axhline(float(row.K), color="gray", ls="--", lw=1, label=f"K={row.K:g}")
                        ax.scatter(
                            [t_ex], [paths[j, t_ex]], color="crimson", zorder=5, s=50,
                            label=f"exercise day {t_ex}",
                        )
                        ax.set_xlabel("day")
                        ax.set_ylabel("S")
                        ax.set_title(
                            f"{ticker} example path | "
                            f"trade {pd.Timestamp(row.trading_date).date()} | "
                            f"dte={int(row.dte)} | model={res.price:.3f} vs "
                            f"mkt={float(row.option_price):.3f}"
                        )
                        ax.legend(frameon=False, loc="best")
                        fig2.tight_layout()
                    _show_fig(fig2)

            display(Markdown(
                "Results stored in `stopping_results` "
                "(dict keyed by ticker → model_price, error, early_ex_frac, mean_ex_day)."
            ))
        except Exception as exc:
            display(Markdown(f"**Error:** `{type(exc).__name__}: {exc}`"))
        finally:
            _stop_busy["on"] = False


_btn_stop.on_click(_run_optimal_stopping)
display(widgets.VBox([
    widgets.HTML("<b>§6 Optimal stopping — SPY / AAPL / MSFT American calls (LSM)</b>"),
    widgets.HBox([_stop_n_paths, _stop_seed, _btn_stop]),
    _stop_out,
]))




## 7. Reminder

1. **§4:** sliders → **Reestimate** → read parameter tables / rolling charts.
2. **§5:** **Start** → Monte Carlo stock paths + expected vs history (one pair per ticker).
3. **§6:** **Compute stopping** → LSM exercise decision + model vs market on SPY / AAPL / MSFT calls (needs §4).
4. **Restart** (§5) only changes the random seed for path plots.

